# Задача 3. Heapsort

## Ход решения

Сначала строю maxheap в исходном списке, просеивая родительские вершины снизу вверх. Затем меняю корень, то есть максимум кучи, с её последним элементом. Максимум встаёт на своё место в отсортированной части справа. Уменьшаю границу кучи и просеиваю новый корень вниз, выбирая большего ребёнка. Повторяю, пока в куче не останется один элемент.

Построение занимает $O(n)$. Дальше выполняется до $n-1$ извлечений с просеиванием на $O(\log n)$ уровней, поэтому в худшем случае время составляет $O(n\log n)$. Перестановки делаю в исходном списке, дополнительная память — $O(1)$.

## Реализация

In [1]:
def _sift_down_max(arr, root, stop):
    """Восстанавливает maxheap под root в пределах arr[0:stop]."""
    while True:
        left = 2 * root + 1
        if left >= stop:
            return

        right = left + 1
        child = left
        if right < stop and arr[left] < arr[right]:
            child = right

        if not arr[root] < arr[child]:
            return

        arr[root], arr[child] = arr[child], arr[root]
        root = child


def heapsort(arr):
    """Сортирует список по возрастанию на месте и возвращает его."""
    size = len(arr)
    for root in range(size // 2 - 1, -1, -1):
        _sift_down_max(arr, root, size)

    for end in range(size - 1, 0, -1):
        arr[0], arr[end] = arr[end], arr[0]
        _sift_down_max(arr, 0, end)
    return arr

## Пошаговый пример

Для списка `[4, 1, 7, 3, 7, -2]` сначала строим maxheap.

| Корень просеивания | Обмены при спуске | Список после спуска |
| ---: | --- | --- |
| 2 | — | `[4, 1, 7, 3, 7, -2]` |
| 1 | 1 ↔ 4 | `[4, 7, 7, 3, 1, -2]` |
| 0 | 0 ↔ 1 | `[7, 4, 7, 3, 1, -2]` |

Теперь последовательно переносим максимумы. Столбец «Обмены при просеивании» относится к активной части после исключения индекса `end`.

| `end` | Максимум | Список сразу после переноса максимума | Обмены при просеивании | Куча после просеивания | Отсортированный суффикс |
| ---: | ---: | --- | --- | --- | --- |
| 5 | 7 | `[-2, 4, 7, 3, 1, 7]` | 0 ↔ 2 | `[7, 4, -2, 3, 1]` | `[7]` |
| 4 | 7 | `[1, 4, -2, 3, 7, 7]` | 0 ↔ 1, 1 ↔ 3 | `[4, 3, -2, 1]` | `[7, 7]` |
| 3 | 4 | `[1, 3, -2, 4, 7, 7]` | 0 ↔ 1 | `[3, 1, -2]` | `[4, 7, 7]` |
| 2 | 3 | `[-2, 1, 3, 4, 7, 7]` | 0 ↔ 1 | `[1, -2]` | `[3, 4, 7, 7]` |
| 1 | 1 | `[-2, 1, 3, 4, 7, 7]` | — | `[-2]` | `[1, 3, 4, 7, 7]` |

Результат в исходном списке — `[-2, 1, 3, 4, 7, 7]`.

In [2]:
example = [4, 1, 7, 3, 7, -2]
print(f"До сортировки:    {example}")
assert heapsort(example) is example
assert example == [-2, 1, 3, 4, 7, 7]
print(f"После сортировки: {example}")

До сортировки:    [4, 1, 7, 3, 7, -2]
После сортировки: [-2, 1, 3, 4, 7, 7]


## Тесты

Сравниваю содержимое исходного списка после сортировки с эталоном `sorted` и проверяю, что функция вернула именно этот объект.

Использую типичные входы, граничные случаи и строки как пример других сравнимых значений. Перебираю все 1 093 списка длины от 0 до 6 из значений −1, 0 и 1. Дополнительно генерирую 2 000 списков с фиксированным зерном, повторами и отрицательными числами. Отдельно проверяю возрастающий, убывающий и полностью одинаковый списки длины 100 000.

In [3]:
from itertools import product
from random import Random
import sys
import unittest


EXAMPLES = (
    [],
    [42],
    [2, 1],
    [1, 2],
    list(range(15)),
    list(range(15, -1, -1)),
    [5] * 17,
    [0, -3, 2, -3, 0, 2, -1],
    [7, 2, 9, 4, 1, 8, 5, 3],
    [3, 2, 3, 1, 2, 4, 5, 5, 6, 4, -2],
    ["б", "в", "а", "а"],
)


def small_inputs():
    for size in range(7):
        for values in product((-1, 0, 1), repeat=size):
            yield list(values)


def random_inputs():
    rng = Random(314159)
    for _ in range(2_000):
        size = rng.randrange(129)
        yield [rng.randint(-30, 30) for _ in range(size)]


def large_inputs():
    size = 100_000
    return (
        ("ascending", list(range(size))),
        ("descending", list(range(size - 1, -1, -1))),
        ("equal", [5] * size),
    )


class HeapSortTests(unittest.TestCase):
    def check_sort(self, values):
        arr = values.copy()
        self.assertIs(heapsort(arr), arr)
        self.assertEqual(arr, sorted(values))

    def test_examples(self):
        for values in (*EXAMPLES, [4, 1, 7, 3, 7, -2]):
            with self.subTest(values=values):
                self.check_sort(values)

    def test_exhaustive_small_inputs(self):
        for values in small_inputs():
            with self.subTest(values=values):
                self.check_sort(values)

    def test_seeded_random_inputs(self):
        for case, values in enumerate(random_inputs()):
            with self.subTest(case=case):
                self.check_sort(values)

    def test_large_patterns(self):
        for name, values in large_inputs():
            with self.subTest(pattern=name):
                self.check_sort(values)

In [4]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(HeapSortTests)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful()

test_examples (__main__.HeapSortTests.test_examples) ... ok
test_exhaustive_small_inputs (__main__.HeapSortTests.test_exhaustive_small_inputs) ... ok
test_large_patterns (__main__.HeapSortTests.test_large_patterns) ... ok
test_seeded_random_inputs (__main__.HeapSortTests.test_seeded_random_inputs) ... ok

----------------------------------------------------------------------
Ran 4 tests in 1.618s

OK


## Эксперимент

Сравниваю медианы семи запусков в миллисекундах. Для каждого метода есть один прогревочный запуск. На четырёх размерах проверяю случайные числа с повторами, возрастающую и убывающую последовательности, а также списки из равных значений. Случайные входы беру из диапазона `[-n, n]` с фиксированным зерном `20261126`.

На каждом запуске метод получает свежую копию одного и того же входа. Порядок методов сдвигается между раундами. `perf_counter` измеряет только вызов функции. Копирование входа, проверка результата и освобождение возвращённого объекта выполняются за пределами замера. Числа в выводах относятся к сохранённому запуску, а ячейка с экспериментом позволяет повторить замеры.


Для эксперимента перенёс `merge` и `merge_sort` из `homework_6/compare/сompare.ipynb`. Создание списка результата внутри `sorted` и `merge_sort` входит в их время.

In [5]:
def merge(left, right):
    result = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    return result + left[i:] + right[j:]


def merge_sort(arr):
    if len(arr) <= 1:
        return arr.copy()

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

In [6]:
import platform
from statistics import median
from time import perf_counter


def benchmark_inputs():
    rng = Random(20261126)
    for size in (1_000, 4_000, 16_000, 64_000):
        yield "Случайный", [rng.randint(-size, size) for _ in range(size)]
        yield "По возрастанию", list(range(size))
        yield "По убыванию", list(range(size - 1, -1, -1))
        yield "Все равны", [5] * size


def run_benchmarks(algorithms):
    repeats = 7
    rows = []

    for pattern, source in benchmark_inputs():
        expected = sorted(source)
        times = {method: [] for method in algorithms}

        for round_index in range(-1, repeats):
            offset = round_index % len(algorithms)
            order = algorithms[offset:] + algorithms[:offset]
            for method in order:
                arr = source.copy()
                start = perf_counter()
                result = method(arr)
                elapsed_ms = (perf_counter() - start) * 1_000

                assert result == expected
                del result
                if round_index >= 0:
                    times[method].append(elapsed_ms)

        rows.append((len(source), pattern, *(median(times[method]) for method in algorithms)))
    return rows


def print_benchmarks(algorithms, rows):
    print(f"{platform.python_implementation()} {platform.python_version()}, "
          f"{platform.system()} {platform.release()} ({platform.machine()}). Медианы, мс.")
    widths = [max(len(method.__name__), 10) for method in algorithms]
    names = " | ".join(f"{method.__name__:>{width}}" for method, width in zip(algorithms, widths))
    print(f"{'n':>6} | {'Вход':<15} | {names}")
    print("-" * (27 + sum(widths) + 3 * (len(widths) - 1)))
    for size, pattern, *times in rows:
        values = " | ".join(f"{value:>{width}.3f}" for value, width in zip(times, widths))
        print(f"{size:>6} | {pattern:<15} | {values}")

In [7]:
algorithms = (heapsort, sorted, merge_sort)
benchmark_rows = run_benchmarks(algorithms)
print_benchmarks(algorithms, benchmark_rows)

CPython 3.12.14, Windows 11 (AMD64). Медианы, мс.
     n | Вход            |   heapsort |     sorted | merge_sort
---------------------------------------------------------------
  1000 | Случайный       |      4.404 |      0.230 |      5.553
  1000 | По возрастанию  |      2.732 |      0.010 |      2.345
  1000 | По убыванию     |      2.326 |      0.010 |      2.178
  1000 | Все равны       |      0.467 |      0.007 |      2.083
  4000 | Случайный       |     12.206 |      0.669 |     14.797
  4000 | По возрастанию  |     11.774 |      0.027 |      8.998
  4000 | По убыванию     |     10.671 |      0.033 |      8.926
  4000 | Все равны       |      1.929 |      0.027 |      9.871
 16000 | Случайный       |     65.787 |      3.137 |     74.234
 16000 | По возрастанию  |     71.823 |      0.134 |     42.095
 16000 | По убыванию     |     54.684 |      0.131 |     40.838
 16000 | Все равны       |      7.039 |      0.093 |     40.172
 64000 | Случайный       |    344.336 |     15.268 |  

## Выводы

- На случайном входе из 64 000 элементов `heapsort` понадобилось 344,336 мс, а `merge_sort` — 363,084 мс. На таком же возрастающем входе их медианы составили 281,517 и 184,013 мс соответственно.
- На списке из 64 000 равных значений `heapsort` занял 36,397 мс, а `merge_sort` — 196,833 мс. У `heapsort` просеивания в этом случае заканчиваются на первой проверке, а `merge_sort` по-прежнему разбивает и сливает части по уровням рекурсии.
- `sorted` оказался самым быстрым во всех 16 случаях. Например, при 64 000 элементах на случайном входе он занял 15,268 мс, на возрастающем — 0,510 мс. Для `heapsort` дополнительная память составляет $O(1)$, а две другие реализации создают списки результата размера $n$.